# Categorical Cross-Entropy (CCE) — Beginner Masterclass

**Reading:** [GeeksforGeeks — Categorical Cross-Entropy in Multi-Class Classification](https://www.geeksforgeeks.org/deep-learning/categorical-cross-entropy-in-multi-class-classification/) and [Loss Functions in Deep Learning](https://www.geeksforgeeks.org/deep-learning/loss-functions-in-deep-learning/)

This notebook explains CCE with hand calculations, pure Python, NumPy, PyTorch, TensorFlow/Keras, and a small MNIST classification experiment.

## Learning goals
- Understand logits, softmax, probabilities, one-hot targets, and categorical cross-entropy.
- Calculate CCE from scratch.
- Know the difference between categorical and sparse categorical cross-entropy.
- Use the correct model output and target format for each framework.
- Avoid double-softmax, incorrect label shapes, and wrong target dtypes.

## 1. What is categorical cross-entropy?

Categorical Cross-Entropy is usually used when each example belongs to **exactly one of more than two classes**. The model predicts a probability distribution over all classes.

For a single sample with \(C\) classes:

\[
\boxed{L=-\sum_{j=1}^{C}y_j\log(p_j)}
\]

Here \(y_j\) is a one-hot target and \(p_j\) is the predicted probability for class \(j\). Since only one target entry is 1, the formula simplifies to:

\[
\boxed{L=-\log(p_{\text{true class}})}
\]

For a batch, average the individual sample losses when using mean reduction. A high loss means the model assigned too little probability to the correct class.

## 2. Softmax turns logits into probabilities

A network usually outputs raw class scores called **logits**. Softmax converts these to probabilities:

\[
p_j=\frac{e^{z_j}}{\sum_{k=1}^{C}e^{z_k}}.
\]

For numerical stability, subtract the largest logit before exponentiating; the resulting softmax is mathematically unchanged.

Example logits:

\[
z=[2.0,1.0,0.1].
\]

For a one-hot target \([1,0,0]\), the first class is correct. We'll calculate the exponential values, probabilities, individual terms, and loss.

In [ ]:
# Pure Python only
import math

logits = [2.0, 1.0, 0.1]
y_one_hot = [1, 0, 0]
max_logit = max(logits)
exp_values = [math.exp(z - max_logit) for z in logits]
sum_exp = sum(exp_values)
probabilities = [v / sum_exp for v in exp_values]
terms = [-y * math.log(max(p, 1e-12)) for y, p in zip(y_one_hot, probabilities)]
cce_manual = sum(terms)

print("Logits:", logits)
print("Maximum logit:", max_logit)
print("Exponentials after max subtraction:", [round(x, 6) for x in exp_values])
print("Sum of exponentials:", round(sum_exp, 6))
print("Softmax probabilities:", [round(x, 6) for x in probabilities])
print("Probability sum:", sum(probabilities))
print("One-hot label:", y_one_hot)
print("Per-class -y_j log(p_j) terms:", [round(x, 6) for x in terms])
print("CCE:", cce_manual)
print("Equivalent -log(p_true_class):", -math.log(probabilities[0]))

## 3. Why CCE penalizes confident mistakes

If the true class receives probability 0.9, its loss is \(-\log(0.9)\approx0.105\). If it receives probability 0.01, the loss is \(-\log(0.01)\approx4.605\).

This makes CCE strongly discourage models that assign very low probability to the correct class.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_true = np.linspace(0.001, 0.999, 500)
plt.figure(figsize=(8, 4))
plt.plot(p_true, -np.log(p_true))
plt.xlabel("Probability assigned to the true class")
plt.ylabel("CCE for one example")
plt.title("Categorical cross-entropy = -log(p_true)")
plt.grid(alpha=.25)
plt.show()

for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"p_true={p:>4}: CCE={-np.log(p):.4f}")

## 4. NumPy: one-hot labels and batch CCE

For \(N\) examples and \(C\) classes:
- target shape: `(N, C)` for one-hot categorical labels.
- probability shape: `(N, C)` where each row sums to 1.

The loss for each sample is \(-\sum_j y_{ij}\log(p_{ij})\). The batch loss here is the mean of per-sample losses.

In [ ]:
y_true_np = np.array([[1,0,0], [0,1,0], [0,0,1]], dtype=np.float64)
probs_np = np.array([[0.7,0.2,0.1], [0.1,0.8,0.1], [0.2,0.2,0.6]], dtype=np.float64)

assert np.allclose(y_true_np.sum(axis=1), 1.0)
assert np.allclose(probs_np.sum(axis=1), 1.0)

per_sample = -np.sum(y_true_np * np.log(np.clip(probs_np, 1e-12, 1.0)), axis=1)
cce_numpy = per_sample.mean()
print("Per-sample CCE:", per_sample)
print("Mean CCE:", cce_numpy)

## 5. PyTorch `CrossEntropyLoss` expects raw logits

Important API rule:

- PyTorch `nn.CrossEntropyLoss` takes **raw logits**, not probabilities.
- The common target format is integer class IDs with dtype `torch.long`.
- Do not apply softmax before `CrossEntropyLoss`; the implementation combines log-softmax and negative log likelihood in a numerically stable way.

We can reproduce the probability example by taking the logarithm of each valid probability: softmax of those log probabilities returns the same probability distribution.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

logits_t = torch.tensor(np.log(probs_np), dtype=torch.float32)
class_ids_t = torch.tensor([0, 1, 2], dtype=torch.long)
loss_torch = nn.CrossEntropyLoss()(logits_t, class_ids_t)

print("PyTorch CrossEntropyLoss:", loss_torch.item())
print("NumPy CCE:", cce_numpy)
print("Softmax(log(probabilities)):\n", F.softmax(logits_t, dim=1).numpy())
assert np.isclose(loss_torch.item(), cce_numpy, atol=1e-5)

## 6. Keras: categorical vs sparse categorical cross-entropy

Keras offers:
- `CategoricalCrossentropy`: use one-hot target vectors.
- `SparseCategoricalCrossentropy`: use integer class IDs.

Both support `from_logits=True` or `False`, which must match model output:
- Softmax probability output → `from_logits=False`.
- Raw logits output → `from_logits=True`.

For stable training, it is often convenient to leave the final model layer as raw logits and use `from_logits=True`.

In [ ]:
import tensorflow as tf

onehot_tf = tf.constant(y_true_np, dtype=tf.float32)
probs_tf = tf.constant(probs_np, dtype=tf.float32)
logits_tf = tf.math.log(probs_tf)
class_ids_tf = tf.constant([0,1,2], dtype=tf.int32)

cce_prob = tf.keras.losses.CategoricalCrossentropy(from_logits=False)(onehot_tf, probs_tf)
cce_logits = tf.keras.losses.CategoricalCrossentropy(from_logits=True)(onehot_tf, logits_tf)
sparse_cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)(class_ids_tf, probs_tf)

print("Keras CCE (probabilities):", float(cce_prob.numpy()))
print("Keras CCE (logits):", float(cce_logits.numpy()))
print("Keras sparse CCE:", float(sparse_cce.numpy()))
assert np.isclose(float(cce_prob.numpy()), cce_numpy, atol=1e-5)
assert np.isclose(float(sparse_cce.numpy()), cce_numpy, atol=1e-5)

## 7. Automatically download MNIST

MNIST consists of 28×28 grayscale handwritten digit images with labels 0–9. Keras downloads it automatically on the first run.

We'll scale pixel values to \([0,1]\), one-hot encode the labels for categorical cross-entropy, and use a subset so training finishes faster in Colab.

In [ ]:
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.mnist.load_data()

x_train = x_train_full[:10000].astype("float32") / 255.0
x_test = x_test_full[:2000].astype("float32") / 255.0
y_train_int = y_train_full[:10000]
y_test_int = y_test_full[:2000]
num_classes = 10
y_train_oh = tf.keras.utils.to_categorical(y_train_int, num_classes)
y_test_oh = tf.keras.utils.to_categorical(y_test_int, num_classes)

print("Training images:", x_train.shape, "| one-hot labels:", y_train_oh.shape)
print("Test images:", x_test.shape, "| one-hot labels:", y_test_oh.shape)
print("First integer target:", y_train_int[0])
print("First one-hot target:", y_train_oh[0])

In [ ]:
plt.figure(figsize=(9,4))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label {y_train_int[i]}")
    plt.axis("off")
plt.suptitle("MNIST samples")
plt.tight_layout()
plt.show()

## 8. Train a digit classifier using CCE

Network:

\[
28\times28\rightarrow\text{Flatten}\rightarrow128\text{ ReLU}\rightarrow64\text{ ReLU}\rightarrow10\text{ logits}
\]

We intentionally do not put softmax in the final layer. `CategoricalCrossentropy(from_logits=True)` handles logits correctly and stably.

In [ ]:
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(28,28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10)  # raw logits
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")]
)
history = model.fit(
    x_train, y_train_oh,
    validation_data=(x_test, y_test_oh),
    epochs=3,
    batch_size=128,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history["loss"], marker="o", label="Train CCE")
plt.plot(history.history["val_loss"], marker="o", label="Test CCE")
plt.xlabel("Epoch"); plt.ylabel("CCE"); plt.title("Training loss"); plt.legend(); plt.grid(alpha=.25)
plt.subplot(1,2,2)
plt.plot(np.array(history.history["accuracy"])*100, marker="o", label="Train")
plt.plot(np.array(history.history["val_accuracy"])*100, marker="o", label="Test")
plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)"); plt.title("Accuracy"); plt.legend(); plt.grid(alpha=.25)
plt.tight_layout(); plt.show()

In [ ]:
logits = model.predict(x_test[:1], verbose=0)
probabilities = tf.nn.softmax(logits, axis=1).numpy()[0]
predicted = int(np.argmax(probabilities))
true_label = int(y_test_int[0])
plt.figure(figsize=(3,3))
plt.imshow(x_test[0], cmap="gray")
plt.title(f"True={true_label}, predicted={predicted}")
plt.axis("off")
plt.show()
print("Class probabilities:", np.round(probabilities, 4))
print("Sum of probabilities:", probabilities.sum())
print("CCE for this example:", -np.log(max(probabilities[true_label], 1e-12)))

## 9. Common mistakes and debugging

- **Softmax applied twice:** PyTorch `CrossEntropyLoss` expects raw logits.
- **Wrong target format:** Keras categorical CCE expects one-hot vectors; sparse CCE expects integer class IDs.
- **Wrong PyTorch dtype:** integer class targets should normally use `torch.long`.
- **Shape mismatch:** logits are commonly `(batch_size, num_classes)` and class indices are `(batch_size,)`.
- **Invalid probabilities:** manual probability-based CCE expects each row to contain values in \([0,1]\) summing to 1.
- **Multilabel task:** if multiple labels may be true at once, independent sigmoid outputs with BCE are usually more suitable than single-label CCE.

## 10. Quiz

1. Why is CCE low when the true class probability is high? Because the loss is \(-\log(p_{\text{true}})\).
2. What does PyTorch `CrossEntropyLoss` expect? Raw logits and usually integer class targets (`torch.long`).
3. Keras one-hot targets: use categorical or sparse categorical? Categorical.
4. Keras integer labels: use categorical or sparse categorical? Sparse categorical.
5. Should logits be passed through softmax before PyTorch CrossEntropyLoss? No.

## Sources and official documentation

### All six reading links supplied for this series
- [GeeksforGeeks — Loss Functions in Deep Learning](https://www.geeksforgeeks.org/deep-learning/loss-functions-in-deep-learning/)
- [GeeksforGeeks — Mean Squared Error](https://www.geeksforgeeks.org/maths/mean-squared-error/)
- [GeeksforGeeks — MAE in PyTorch](https://www.geeksforgeeks.org/python/how-to-measure-the-mean-absolute-error-mae-in-pytorch/)
- [Medium — Mean Absolute Error](https://medium.com/@20__80__/mean-absolute-error-mae-machine-learning-ml-b9b4afc63077)
- [GeeksforGeeks — Binary Cross-Entropy](https://www.geeksforgeeks.org/deep-learning/binary-cross-entropy-log-loss-for-binary-classification/)
- [GeeksforGeeks — Categorical Cross-Entropy](https://www.geeksforgeeks.org/deep-learning/categorical-cross-entropy-in-multi-class-classification/)

### Official API documentation for implementation/debugging
- [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [Keras probabilistic losses](https://keras.io/api/losses/probabilistic_losses/)
- [TensorFlow/Keras MNIST data](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist/load_data)
- [PyTorch functional cross-entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

The explanations summarize the articles in original wording rather than copying full webpages. When debugging, check target encoding, tensor shapes, dtypes, whether the model returns logits or probabilities, and the official API docs.